# Building the Parquet files for the demo

The data comes from SwissMetNet, the MeteoSwiss weather stations. Each station has one CSV with a value every 10 minutes, on `data.geo.admin.ch/ch.meteoschweiz.ogd-smn`.

In [1]:
import urllib.request
from datetime import datetime, timedelta
from pathlib import Path

import polars as pl

DATA = Path("data")
BASE = "https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn"
PERIOD = "t_recent"  # current year, January 1st to yesterday
ROW_GROUP = 10_000  # rows per row group in the Parquet file
SINCE = datetime.now() - timedelta(days=180)  # Fix the beginning date

print("keeping everything after", SINCE)

keeping everything after 2026-03-28 21:52:21.696263


## 1. One station, from CSV to a clean DataFrame

In [2]:
# MeteoSwiss CSVs are in cp1252, Polars wants UTF-8
def fetch(url):
    with urllib.request.urlopen(url, timeout=180) as r:
        return r.read().decode("cp1252").encode("utf8")

In [3]:
# one station: download the CSV, keep 5 columns over the last 180 days
def measurements(abbr):
    url = f"{BASE}/{abbr}/ogd-smn_{abbr}_{PERIOD}.csv"
    df = pl.read_csv(fetch(url), separator=";", infer_schema_length=0)  # every column read as text

    # some stations have no barometer (or thermometer), add the missing column empty
    for code in ("tre200s0", "ure200s0", "prestas0"):
        if code not in df.columns:
            df = df.with_columns(pl.lit(None).alias(code))

    df = df.select(
        pl.col("station_abbr").str.to_lowercase(),
        pl.col("reference_timestamp").str.to_datetime("%d.%m.%Y %H:%M").alias("ts"),
        pl.col("tre200s0").cast(pl.Float64, strict=False).alias("temperature"),  # air temperature at 2 m
        pl.col("ure200s0").cast(pl.Float64, strict=False).alias("humidity"),  # relative humidity at 2 m
        pl.col("prestas0").cast(pl.Float64, strict=False).alias("pressure"),  # pressure at the station
    )
    return df.filter(pl.col("ts") >= SINCE)

## 2. The station reference table

In [4]:
# read the CSV into a Polars DataFrame
raw_stations = pl.read_csv(fetch(f"{BASE}/ogd-smn_meta_stations.csv"), separator=";",
                           infer_schema_length=None)   # None tells Polars to look at every row to guess the column types

print(f"{raw_stations.height} stations x {raw_stations.width} columns")
print(raw_stations.columns)
raw_stations.head(3)

158 stations x 24 columns
['station_abbr', 'station_name', 'station_canton', 'station_wigos_id', 'station_type_de', 'station_type_fr', 'station_type_it', 'station_type_en', 'station_dataowner', 'station_data_since', 'station_height_masl', 'station_height_barometer_masl', 'station_coordinates_lv95_east', 'station_coordinates_lv95_north', 'station_coordinates_wgs84_lat', 'station_coordinates_wgs84_lon', 'station_exposition_de', 'station_exposition_fr', 'station_exposition_it', 'station_exposition_en', 'station_url_de', 'station_url_fr', 'station_url_it', 'station_url_en']


station_abbr,station_name,station_canton,station_wigos_id,station_type_de,station_type_fr,station_type_it,station_type_en,station_dataowner,station_data_since,station_height_masl,station_height_barometer_masl,station_coordinates_lv95_east,station_coordinates_lv95_north,station_coordinates_wgs84_lat,station_coordinates_wgs84_lon,station_exposition_de,station_exposition_fr,station_exposition_it,station_exposition_en,station_url_de,station_url_fr,station_url_it,station_url_en
str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str
"""ABO""","""Adelboden""","""BE""","""0-20000-0-06735""","""Automatische Wetterstationen -…","""Stations météorologiques autom…","""Stazioni meteorologiche automa…","""Automatic weather stations - M…","""MeteoSchweiz, SLF""","""01.01.1901""",1321.0,1326.0,2.609372e6,1.148939e6,46.491703,7.560703,"""Südosthang""","""Versant sud-est""","""Versante sud-orientale""","""south-eastern oriented slope""","""https://www.meteoschweiz.admin…","""https://www.meteosuisse.admin.…","""https://www.meteosvizzera.admi…","""https://www.meteoswiss.admin.c…"
"""AEG""","""Oberägeri""","""ZG""","""0-20000-0-06676""","""Automatische Wetterstationen -…","""Stations météorologiques autom…","""Stazioni meteorologiche automa…","""Automatic weather stations - M…","""MeteoSchweiz""","""01.02.1993""",724.0,null,2.688729e6,1.220956e6,47.133636,8.608206,"""Ebene""","""Plaine""","""Pianura""","""plain""","""https://www.meteoschweiz.admin…","""https://www.meteosuisse.admin.…","""https://www.meteosvizzera.admi…","""https://www.meteoswiss.admin.c…"
"""AIG""","""Aigle""","""VD""","""0-20000-0-06712""","""Automatische Wetterstationen -…","""Stations météorologiques autom…","""Stazioni meteorologiche automa…","""Automatic weather stations - M…","""MeteoSchweiz""","""01.01.1961""",381.0,382.0,2.560404e6,1.130713e6,46.326647,6.924472,"""Ebene""","""Plaine""","""Pianura""","""plain""","""https://www.meteoschweiz.admin…","""https://www.meteosuisse.admin.…","""https://www.meteosvizzera.admi…","""https://www.meteoswiss.admin.c…"


In [5]:
# the station table, saved later as stations.parquet
# Polars eager: select and sort
stations = raw_stations.select(
    pl.col("station_abbr").str.to_lowercase(),
    pl.col("station_name"),
    pl.col("station_canton").alias("canton"),
    pl.col("station_height_masl").cast(pl.Int64).alias("height_masl"),
).sort("station_abbr")

stations.head(5)

station_abbr,station_name,canton,height_masl
str,str,str,i64
"""abo""","""Adelboden""","""BE""",1321
"""aeg""","""Oberägeri""","""ZG""",724
"""aig""","""Aigle""","""VD""",381
"""alt""","""Altdorf""","""UR""",438
"""and""","""Andeer""","""GR""",987


In [6]:
station_list = stations["station_abbr"].to_list()
print(len(station_list), "stations to download")
print(station_list[:12], "...")

158 stations to download
['abo', 'aeg', 'aig', 'alt', 'and', 'ant', 'arh', 'aro', 'att', 'ban', 'bas', 'beh'] ...


## 3. Download loop

In [ ]:
frames, dropped = [], []
for abbr in station_list:
    try:
        d = measurements(abbr)
    except Exception as exc:
        dropped.append((abbr, type(exc).__name__))  # e.g. ("bla", "HTTPError")
        continue
    if d.height == 0:
        dropped.append((abbr, "no data in the last 180 days"))  # CSV has no data in the last 180 days
    elif d["temperature"].null_count() == d.height:
        dropped.append((abbr, "no temperature"))  # all temperatures null -> no thermometer
    else:
        print(f"  {abbr}: {d.height:,} rows")
        frames.append(d)  # if everything is all right, append to the frames

print(f"\n{len(frames)} kept, {len(dropped)} dropped: {dropped}")  # summary

  abo: 25,788 rows
  aig: 25,788 rows
  alt: 25,788 rows
  and: 25,788 rows
  ant: 25,788 rows
  arh: 25,788 rows
  aro: 25,788 rows
  att: 25,788 rows
  bas: 25,788 rows
  beh: 25,788 rows
  ber: 25,788 rows
  bez: 25,788 rows
  bia: 25,788 rows
  bie: 25,788 rows
  bin: 25,788 rows
  biv: 25,788 rows
  biz: 25,788 rows
  bla: 18,390 rows
  bol: 25,788 rows
  bou: 25,788 rows
  brl: 25,788 rows
  buf: 25,788 rows
  bus: 25,788 rows
  cdf: 25,788 rows
  cdm: 25,788 rows
  cev: 25,788 rows
  cgi: 25,788 rows
  cha: 25,788 rows
  chb: 25,788 rows
  chd: 25,788 rows
  chm: 25,788 rows
  chu: 25,788 rows
  chz: 25,788 rows
  cim: 25,788 rows
  cma: 25,788 rows
  com: 25,788 rows
  cov: 25,788 rows
  coy: 25,593 rows
  crm: 25,788 rows
  dav: 25,788 rows
  dem: 25,788 rows


## 4. Concatenate and sort

In [ ]:
# Polars eager: concat all stations, sort by station so the Parquet row groups are split by station
all_measurements = pl.concat(frames).sort("station_abbr", "ts")

kept = all_measurements["station_abbr"].unique().to_list()  # used in the last cell for stations.parquet
print(f"{all_measurements.height:,} rows, {len(kept)} stations")

## 5. Write the Parquet files

In [ ]:
DATA.mkdir(exist_ok=True)

# write in row groups of 10 000 rows, each with its min/max, so Polars can skip the ones without ber
OUT = DATA / "measurements.parquet"
all_measurements.write_parquet(OUT, row_group_size=ROW_GROUP, statistics=True)

# the station table, only the stations that have measurements
(
    stations
    .filter(pl.col("station_abbr").is_in(kept))
    .write_parquet(DATA / "stations.parquet")
)

print(f"{OUT}: {OUT.stat().st_size / 1e6:.1f} MB")